## Cell 1 - English text

English text,English text LM Studio English text,English text Phase 4 English text:English text/English text, English text Planner, Optimizer, PairwiseBattle English text SelfCorrector.


In [1]:
import asyncio
import os
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / ".env")

from src.core.model_provider import GenerateParams, GenerateResult, StructuredParams, TokenUsage
from src.core.providers.google_provider import GoogleProvider
from src.core.providers.lmstudio_provider import LMStudioProvider
from src.core.providers.openai_provider import OpenAIProvider
from src.core.token_estimator import TokenEstimator
from src.core.types import (
    EvaluationResult,
    EvaluationScores,
    JudgeVote,
    OptimizationConstraints,
    OptimizationResult,
    PlanningResult,
    TokenStats,
)
from src.evaluation.judge_pool import JudgePool
from src.evaluation.pairwise_battle import PairwiseBattle
from src.evaluation.self_correction import SelfCorrector
from src.optimization.optimizer import Optimizer
from src.planning.planner import Planner

LM_STUDIO_URL = os.getenv("LM_STUDIO_URL", "http://localhost:1234/v1").rstrip("/")
te = TokenEstimator()
evaluation_rows = []
artifacts = {}

# English text LM Studio English text,English text ID.
available_models = []
try:
    import openai

    lmstudio_client = openai.OpenAI(base_url=LM_STUDIO_URL, api_key="lm-studio")
    available_models = [model.id for model in lmstudio_client.models.list().data]
except Exception as exc:
    print(f"English text LM Studio models.list(): {type(exc).__name__}: {exc}")

print("English text LM Studio English text:", available_models)
MODEL_ID = available_models[0] if available_models else None
print("English text:", MODEL_ID or "English text,English text/English text")

lmstudio_provider = None
planner = None
optimizer = None
if MODEL_ID:
    try:
        lmstudio_provider = LMStudioProvider(model=MODEL_ID, base_url=LM_STUDIO_URL)
        planner = Planner(
            model_provider=lmstudio_provider,
            refs_dir=str(PROJECT_ROOT / "src/planning/instruction_refs"),
        )
        optimizer = Optimizer(
            model_provider=lmstudio_provider,
            token_estimator=te,
            constraints=OptimizationConstraints(
                maxCompressionRate=0.50,
                minQualityScore=6.0,
                requirePositiveROI=True,
                maxSelfCorrectionRetries=2,
            ),
        )
        print("Planner English text Optimizer English text.")
    except Exception as exc:
        print(f"English text LM Studio Provider/Planner/Optimizer English text: {type(exc).__name__}: {exc}")

openai_provider = None
google_provider = None
try:
    openai_provider = OpenAIProvider(model="gpt-4o")
    print("OpenAIProvider(gpt-4o) English text.")
except Exception as exc:
    print(f"OpenAIProvider English text: {type(exc).__name__}: {exc}")

try:
    google_provider = GoogleProvider(model="gemini-2.5-flash")
    print("GoogleProvider(gemini-2.5-flash) English text.")
except Exception as exc:
    print(f"GoogleProvider English text: {type(exc).__name__}: {exc}")

evaluation_providers = [provider for provider in [openai_provider, google_provider] if provider is not None]
judge_pool = JudgePool(evaluation_providers) if evaluation_providers else None
battle = (
    PairwiseBattle(
        target_providers=evaluation_providers,
        judge_pool=judge_pool,
        constraints=OptimizationConstraints(minQualityScore=6.0, maxSelfCorrectionRetries=2),
    )
    if judge_pool and evaluation_providers
    else None
)
self_corrector = (
    SelfCorrector(optimizer=optimizer, battle=battle, constraints=OptimizationConstraints(maxSelfCorrectionRetries=2))
    if optimizer and battle
    else None
)

print("PairwiseBattle English text:", "English text" if battle else "English text(English text API provider)")
print("SelfCorrector English text:", "English text" if self_corrector else "English text(English text)")


English text LM Studio English text: ['gemma-4-e4b-it', 'gpt-oss-nano', 'qwen3.5-4b', 'qwen3.5-9b-uncensored-hauhaucs-aggressive', 'qwen3.5-9b-abliterated', 'text-embedding-nomic-embed-text-v1.5']
English text: gemma-4-e4b-it
Planner English text Optimizer English text.
OpenAIProvider(gpt-4o) English text.


GoogleProvider(gemini-2.5-flash) English text.
PairwiseBattle English text: English text
SelfCorrector English text: English text


## Cell 2 - English text(EB-01)

English text vibe coding prompt English text:Planning -> Optimization -> Pairwise Battle,English text winner, English text.


In [2]:
print("=== EB-01: English text vibe coding prompt English text ===")
raw_prompt_eb01 = "English text"

try:
    if not all([planner, optimizer, battle]):
        raise RuntimeError("EB-01 English text planner, optimizer English text battle English text.")

    planning_eb01 = await planner.plan(raw_prompt_eb01)
    optimization_eb01 = await optimizer.optimize(raw_prompt_eb01, planning_eb01)
    result_eb01 = await battle.evaluate(
        original_prompt=raw_prompt_eb01,
        optimized_prompt=optimization_eb01.optimizedPrompt,
        task_description=planning_eb01.detectedIntent,
    )

    print("winner:", result_eb01.winner)
    print("\nJudgeVote:")
    for vote in result_eb01.judgeResults:
        print(f"- model={vote.model} winner={vote.winner} reasoning={vote.reasoning}")

    print("\nEvaluationScores:")
    print(f"intentAlignment:  {result_eb01.scores.intentAlignment:.2f}")
    print(f"logicCoherence:   {result_eb01.scores.logicCoherence:.2f}")
    print(f"concisenessScore: {result_eb01.scores.concisenessScore:.2f}")
    print(f"formatCompliance: {result_eb01.scores.formatCompliance:.2f}")
    print(f"overall:          {result_eb01.scores.overall:.2f}")

    artifacts["EB-01"] = {
        "raw_prompt": raw_prompt_eb01,
        "planning": planning_eb01,
        "optimization": optimization_eb01,
        "result": result_eb01,
        "retries": 0,
    }
    evaluation_rows.append({
        "prompt_id": "EB-01",
        "original_tokens": optimization_eb01.tokenStats.originalCount,
        "optimized_tokens": optimization_eb01.tokenStats.optimizedCount,
        "winner": result_eb01.winner,
        "overall_score": result_eb01.scores.overall,
        "retries": 0,
    })
except Exception as exc:
    print(f"EB-01 English text: {type(exc).__name__}: {exc}")
    evaluation_rows.append({
        "prompt_id": "EB-01",
        "original_tokens": te.exact_count(raw_prompt_eb01),
        "optimized_tokens": None,
        "winner": "skipped",
        "overall_score": None,
        "retries": 0,
    })


=== EB-01: English text vibe coding prompt English text ===


winner: optimized

JudgeVote:
- model=openai:gpt-4o winner=tie reasoning=Both responses provide a complete and functional implementation of a login feature using Python and Flask. They include session management, user authentication, and HTML templates for the login form. Both responses are logically coherent and align well with the task intent. They are concise, with minimal redundancy, and follow the expected format. The differences are minor, such as the use of message categories in response B, but these do not significantly impact the overall quality.
- model=google:gemini-2.5-flash winner=optimized reasoning=Both responses provide excellent, complete, and functional Flask login implementations. Response B is slightly better due to its more robust handling of Flask's 'flash' messages by utilizing categories, which allows for more flexible UI feedback. Response A's home page link is a nice touch, but B's flash message implementation is a better practice for a more complete applicati

## Cell 3 - English text(EB-02)

Build a deliberately weak optimized prompt and verify that PairwiseBattle forces winner to original when overall is below `minQualityScore`, then records feedback.


In [3]:
print("=== EB-02: English text ===")
raw_prompt_eb02 = "English text FastAPI English text,English text, JWT English text, English text JSON English text."
bad_optimized_prompt_eb02 = "English text."

class EB02TargetProvider:
    id = "eb02-target"
    type = "local"

    async def generate(self, params: GenerateParams) -> GenerateResult:
        return GenerateResult(
            text=f"response for {params.prompt[:16]}",
            usage=TokenUsage(prompt_tokens=1, completion_tokens=1),
        )

    async def generate_structured(self, params: StructuredParams, response_type):
        raise NotImplementedError("EB-02 English text structured generation")

    def count_tokens(self, text: str) -> int:
        return te.exact_count(text)

    async def health_check(self):
        raise NotImplementedError("EB-02 English text health_check")

class EB02LowScoreJudgePool:
    async def evaluate(self, original_prompt, optimized_prompt, response_a, response_b, task_description):
        low_scores = EvaluationScores(
            intentAlignment=5.0,
            logicCoherence=5.0,
            concisenessScore=5.0,
            formatCompliance=5.0,
            overall=5.0,
        )
        votes = [
            JudgeVote(model="mock-judge-a", winner="optimized", reasoning="Mock low quality optimized vote."),
            JudgeVote(model="mock-judge-b", winner="optimized", reasoning="Mock low quality optimized vote."),
        ]
        for vote in votes:
            object.__setattr__(vote, "_scores", low_scores)
        return votes

try:
    eb02_battle = PairwiseBattle(
        target_providers=[EB02TargetProvider()],
        judge_pool=EB02LowScoreJudgePool(),
        constraints=OptimizationConstraints(minQualityScore=6.0),
    )
    result_eb02 = await eb02_battle.evaluate(
        original_prompt=raw_prompt_eb02,
        optimized_prompt=bad_optimized_prompt_eb02,
        task_description="English text, English text, English text FastAPI English text.",
    )

    print("winner(English text original):", result_eb02.winner)
    print("feedback:", result_eb02.feedback)
    print("overall:", result_eb02.scores.overall)
    assert result_eb02.winner == "original"
    assert result_eb02.feedback

    evaluation_rows.append({
        "prompt_id": "EB-02",
        "original_tokens": te.exact_count(raw_prompt_eb02),
        "optimized_tokens": te.exact_count(bad_optimized_prompt_eb02),
        "winner": result_eb02.winner,
        "overall_score": result_eb02.scores.overall,
        "retries": 0,
    })
    artifacts["EB-02"] = {
        "raw_prompt": raw_prompt_eb02,
        "optimized_prompt": bad_optimized_prompt_eb02,
        "result": result_eb02,
        "retries": 0,
    }
except Exception as exc:
    print(f"EB-02 English text: {type(exc).__name__}: {exc}")
    evaluation_rows.append({
        "prompt_id": "EB-02",
        "original_tokens": te.exact_count(raw_prompt_eb02),
        "optimized_tokens": te.exact_count(bad_optimized_prompt_eb02),
        "winner": "error",
        "overall_score": None,
        "retries": 0,
    })


=== EB-02: English text ===
winner(English text original): original
feedback: Overall score 5.0 below threshold 6.0
overall: 5.0


## Cell 4 - English text(EB-03)

English text mock English text,English text `SelfCorrector.correct()`,English text winner English text, English text prompt.


In [4]:
print("=== EB-03: English text ===")

class NotebookOptimizerMock:
    def __init__(self):
        self.planning_snapshots = []

    async def optimize(self, raw_prompt: str, planning_result: PlanningResult) -> OptimizationResult:
        self.planning_snapshots.append(list(planning_result.refinedRequirements))
        retry_no = len(self.planning_snapshots)
        return OptimizationResult(
            optimizedPrompt=f"{raw_prompt}\n\n[Self-corrected retry {retry_no}]\n" + "\n".join(planning_result.refinedRequirements),
            tokenStats=TokenStats(
                originalCount=te.exact_count(raw_prompt),
                optimizedCount=te.exact_count(raw_prompt) + 10 * retry_no,
                reductionRate=0.0,
            ),
            appliedTechniques=["mock self-correction"],
        )

class NotebookBattleMock:
    def __init__(self, winners):
        self.winners = winners
        self.history = []

    async def evaluate(self, original_prompt: str, optimized_prompt: str, task_description: str) -> EvaluationResult:
        winner = self.winners[min(len(self.history), len(self.winners) - 1)]
        self.history.append(winner)
        score = 7.2 if winner == "optimized" else 4.8
        return EvaluationResult(
            winner=winner,
            scores=EvaluationScores(
                intentAlignment=4.0 if winner == "original" else 8.0,
                logicCoherence=5.0 if winner == "original" else 7.5,
                concisenessScore=6.0,
                formatCompliance=5.5,
                overall=score,
            ),
            judgeResults=[],
            feedback="mock failure" if winner == "original" else None,
        )

try:
    raw_prompt_eb03 = "English text"
    planning_eb03 = PlanningResult(
        detectedIntent="English text",
        scene="vibe_coding",
        refinedRequirements=["Task: English text", "Format: English text"],
        instructionRefs=["vibe_coding"],
    )
    initial_result_eb03 = EvaluationResult(
        winner="original",
        scores=EvaluationScores(
            intentAlignment=3.0,
            logicCoherence=6.0,
            concisenessScore=6.5,
            formatCompliance=7.0,
            overall=4.9,
        ),
        judgeResults=[],
        feedback="mock initial loss",
    )

    mock_optimizer = NotebookOptimizerMock()
    mock_battle = NotebookBattleMock(winners=["original", "optimized"])
    mock_corrector = SelfCorrector(
        optimizer=mock_optimizer,
        battle=mock_battle,
        constraints=OptimizationConstraints(maxSelfCorrectionRetries=2),
    )

    final_prompt_eb03, final_result_eb03 = await mock_corrector.correct(
        raw_prompt=raw_prompt_eb03,
        planning_result=planning_eb03,
        initial_result=initial_result_eb03,
        task_description="English text",
    )

    print("winner English text:", ["initial:original"] + [f"retry_{i + 1}:{winner}" for i, winner in enumerate(mock_battle.history)])
    print("English text:", len(mock_battle.history))
    print("English text winner:", final_result_eb03.winner)
    print("English text prompt:")
    print(final_prompt_eb03)

    evaluation_rows.append({
        "prompt_id": "EB-03",
        "original_tokens": te.exact_count(raw_prompt_eb03),
        "optimized_tokens": te.exact_count(final_prompt_eb03),
        "winner": final_result_eb03.winner,
        "overall_score": final_result_eb03.scores.overall,
        "retries": len(mock_battle.history),
    })
    artifacts["EB-03"] = {
        "raw_prompt": raw_prompt_eb03,
        "planning": planning_eb03,
        "result": final_result_eb03,
        "final_prompt": final_prompt_eb03,
        "retries": len(mock_battle.history),
    }
except Exception as exc:
    print(f"EB-03 English text: {type(exc).__name__}: {exc}")


=== EB-03: English text ===
winner English text: ['initial:original', 'retry_1:original', 'retry_2:optimized']
English text: 2
English text winner: optimized
English text prompt:
English text

[Self-corrected retry 2]
Task: English text
Format: English text
Focus strictly on the core task. Remove any content not directly related to the objective.
Focus strictly on the core task. Remove any content not directly related to the objective.


## Cell 5 - English text DataFrame

English text EB-01 English text EB-03 English text,English text token, winner, overall English text.


In [5]:
print("=== EB-01 ~ EB-03 English text ===")
summary_df = pd.DataFrame(
    evaluation_rows,
    columns=[
        "prompt_id",
        "original_tokens",
        "optimized_tokens",
        "winner",
        "overall_score",
        "retries",
    ],
)

display(summary_df)


=== EB-01 ~ EB-03 English text ===


,prompt_id,original_tokens,optimized_tokens,winner,overall_score,retries
0,EB-01,7,7,optimized,9.55,0
1,EB-02,32,3,original,5.00,0
2,EB-03,7,70,optimized,7.20,2


## Cell 6 - English text(English text API)

English text mock provider English text mock judge pool English text Pairwise Battle English text, English text, English text,English text SelfCorrector English text.English text.


In [6]:
print("=== English text:Pairwise Battle + SelfCorrector ===")

class OfflineTargetProvider:
    id = "offline-target"
    type = "local"

    async def generate(self, params: GenerateParams) -> GenerateResult:
        return GenerateResult(
            text=f"response for: {params.prompt[:24]}",
            usage=TokenUsage(prompt_tokens=1, completion_tokens=1),
        )

    async def generate_structured(self, params: StructuredParams, response_type):
        raise NotImplementedError("OfflineTargetProvider English text structured generation")

    def count_tokens(self, text: str) -> int:
        return len(text.split())

    async def health_check(self):
        raise NotImplementedError("OfflineTargetProvider English text health_check")

class OfflineJudgePool:
    def __init__(self, votes):
        self.votes = votes

    async def evaluate(self, original_prompt, optimized_prompt, response_a, response_b, task_description):
        return self.votes

class OfflineOptimizer:
    def __init__(self):
        self.calls = []

    async def optimize(self, raw_prompt: str, planning_result: PlanningResult) -> OptimizationResult:
        self.calls.append(list(planning_result.refinedRequirements))
        return OptimizationResult(
            optimizedPrompt=f"optimized retry {len(self.calls)}",
            tokenStats=TokenStats(originalCount=10, optimizedCount=8, reductionRate=0.2),
        )

class OfflineFailingBattle:
    def __init__(self):
        self.calls = []

    async def evaluate(self, original_prompt: str, optimized_prompt: str, task_description: str) -> EvaluationResult:
        self.calls.append(optimized_prompt)
        return make_result("original", overall=5.5, intent=3.0, logic=6.0, concise=6.0, fmt=6.0)

def make_scores(intent=8.0, logic=8.0, concise=8.0, fmt=8.0, overall=None):
    if overall is None:
        overall = intent * 0.4 + logic * 0.3 + concise * 0.2 + fmt * 0.1
    return EvaluationScores(
        intentAlignment=intent,
        logicCoherence=logic,
        concisenessScore=concise,
        formatCompliance=fmt,
        overall=overall,
    )

def make_vote(model: str, winner: str, scores: EvaluationScores) -> JudgeVote:
    vote = JudgeVote(model=model, winner=winner, reasoning=f"{model} chose {winner}")
    object.__setattr__(vote, "_scores", scores)
    return vote

def make_result(winner: str, overall=8.0, intent=8.0, logic=8.0, concise=8.0, fmt=8.0) -> EvaluationResult:
    return EvaluationResult(
        winner=winner,
        scores=make_scores(intent=intent, logic=logic, concise=concise, fmt=fmt, overall=overall),
        judgeResults=[],
        feedback="offline failure" if winner == "original" else None,
    )

async def run_offline_checks():
    targets = [OfflineTargetProvider()]

    # 1 English text:2 English text optimized + 1 English text original -> winner = optimized
    majority_votes = [
        make_vote("judge-a", "optimized", make_scores()),
        make_vote("judge-b", "optimized", make_scores()),
        make_vote("judge-c", "original", make_scores()),
    ]
    majority_battle = PairwiseBattle(
        target_providers=targets,
        judge_pool=OfflineJudgePool(majority_votes),
        constraints=OptimizationConstraints(minQualityScore=6.0),
    )
    majority_result = await majority_battle.evaluate("original", "optimized", "task")
    assert majority_result.winner == "optimized"

    # 2 English text:1 optimized + 1 original -> winner = optimized
    tie_votes = [
        make_vote("judge-a", "optimized", make_scores()),
        make_vote("judge-b", "original", make_scores()),
    ]
    tie_battle = PairwiseBattle(
        target_providers=targets,
        judge_pool=OfflineJudgePool(tie_votes),
        constraints=OptimizationConstraints(minQualityScore=6.0),
    )
    tie_result = await tie_battle.evaluate("original", "optimized", "task")
    assert tie_result.winner == "optimized"

    # 3 English text:overall = 5.0 < 6.0 -> winner English text original
    low_quality_votes = [
        make_vote("judge-a", "optimized", make_scores(intent=5, logic=5, concise=5, fmt=5, overall=5.0)),
        make_vote("judge-b", "optimized", make_scores(intent=5, logic=5, concise=5, fmt=5, overall=5.0)),
    ]
    gated_battle = PairwiseBattle(
        target_providers=targets,
        judge_pool=OfflineJudgePool(low_quality_votes),
        constraints=OptimizationConstraints(minQualityScore=6.0),
    )
    gated_result = await gated_battle.evaluate("original", "optimized", "task")
    assert gated_result.winner == "original"
    assert gated_result.feedback

    # 4 SelfCorrector English text:mock English text -> English text 2 English text raw_prompt
    planning = PlanningResult(
        detectedIntent="offline task",
        scene="vibe_coding",
        refinedRequirements=["Task: offline task"],
        instructionRefs=["vibe_coding"],
    )
    offline_optimizer = OfflineOptimizer()
    offline_battle = OfflineFailingBattle()
    corrector = SelfCorrector(
        optimizer=offline_optimizer,
        battle=offline_battle,
        constraints=OptimizationConstraints(maxSelfCorrectionRetries=2),
    )
    raw_prompt = "raw_prompt"
    initial_result = make_result("original", overall=5.0, intent=2.0, logic=6.0, concise=6.0, fmt=6.0)
    final_prompt, final_result = await corrector.correct(raw_prompt, planning, initial_result, "task")
    assert final_prompt == raw_prompt
    assert len(offline_optimizer.calls) == 2
    assert len(offline_battle.calls) == 2
    assert final_result.feedback == "Self-correction exhausted after 2 retries"

await run_offline_checks()
print("=== English text ===")


=== English text:Pairwise Battle + SelfCorrector ===
=== English text ===
